# Common-mean gamma simulation with 1% contamination

This notebook calibrates the DPD tuning parameter using the
**clean-precision-refined, efficiency- and target-preservation-constrained
one-standard-error rule**.

For every candidate \(\tau\), paired clean and contaminated training datasets
estimate

\[
\widehat R_c(\tau)
=
\frac{1}{R_{\mathrm{train}}}
\sum_{r=1}^{R_{\mathrm{train}}}
\left(
\widehat\mu_{r,c,\tau}-\mu_0
\right)^2,
\qquad
c\in\{\mathrm{clean},\mathrm{cont}\}.
\]

The clean relative-MSE efficiency is

\[
\widehat E_{\mathrm{clean}}(\tau)
=
\frac{\widehat R_{\mathrm{clean}}(0)}
{\widehat R_{\mathrm{clean}}(\tau)}.
\]

The simulation target-drift measure is

\[
\widehat D_{\mathrm{MC}}(\tau)
=
\frac{
\left|
\overline{\widehat\mu}_{\mathrm{clean},\tau}
-
\overline{\widehat\mu}_{\mathrm{clean},0}
\right|
}{
\left|
\overline{\widehat\mu}_{\mathrm{clean},0}
\right|
}.
\]

A candidate is feasible when

\[
\widehat E_{\mathrm{clean}}(\tau)\ge\eta,
\qquad
\widehat q(\tau)\ge q_0,
\qquad
\widehat D_{\mathrm{MC}}(\tau)\le\delta.
\]

After finding the feasible contaminated-risk minimizer
\(\widehat\tau_{\min}\), the robust one-standard-error set contains all
feasible values satisfying

\[
\widehat R_{\mathrm{cont}}(\tau)
\le
\widehat R_{\mathrm{cont}}(\widehat\tau_{\min})
+
\widehat{\operatorname{MCSE}}_{\mathrm{cont}}
(\widehat\tau_{\min}).
\]

The final tuning value is selected lexicographically by

\[
\widehat\tau_{\mathrm{PR\text{-}1SE}}
=
\operatorname*{lex\,argmin}
\left(
\widehat s_{\mathrm{clean}}(\tau),
\widehat D_{\mathrm{MC}}(\tau),
\tau
\right)
\]

over the robust one-standard-error set. It is frozen before independent clean
and contaminated test datasets are generated.

The data-generating design is

\[
\mu_0=10,\quad
(\alpha_1,\alpha_2)=(0.5,5),\quad
(n_1,n_2)=(500,1500),
\]

with each observation multiplied by \(10\) independently with probability
\(0.01\).


In [1]:
"""
ADPD tau calibration by a clean-precision-refined, efficiency- and target-preservation-constrained one-SE rule,
followed by an independent comparison of the other common-mean
estimators using their own absolute performance measures.

Estimators reported on the held-out test simulations
-----------------------------------------------------
1. Equal
2. Pooled
3. BetterComponentPlugIn
4. Plugin
5. MOM
6. MLE
7. ADPD-OneSE

No estimator is expressed as a ratio, difference, or improvement relative
to MLE. MLE remains one estimator in the ordinary comparison table.

Tau selection
-------------
Only the ADPD family is used to calibrate tau. For each candidate tau,
paired clean and contaminated training datasets are used to estimate

    clean_MSE_efficiency(tau)
        = MSE_clean(tau=0) / MSE_clean(tau),

and

    MSE_cont(tau).

The feasible set is

    F = {tau:
         clean_MSE_efficiency(tau) >= CLEAN_EFFICIENCY_FLOOR
         and fit_success_rate(tau) >= MIN_FIT_SUCCESS_RATE}.

Let tau_min be the candidate in F with the smallest contaminated MSE.
The robust one-standard-error set contains every feasible tau satisfying

    MSE_cont(tau)
        <= MSE_cont(tau_min) + MCSE[MSE_cont(tau_min)].

Within that set, tau is selected by the smallest clean bootstrap standard deviation, then the smallest target drift, and then the smallest tau. The selected value is frozen before any held-out test dataset is used.
All seven estimators are then summarized independently on paired clean and
contaminated test datasets using mean estimate, bias, variance, MSE, RMSE,
MAE, fit-success rate, and within-scenario MSE rank.

Interpretation
--------------
This procedure calibrates a scenario-specific fixed ADPD tuning value.
Tau is not a parameter of the data-generating distribution.
"""

from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import gammaln


# ============================================================
# 1. Configuration
# ============================================================

SEED = 12345

TRUE_MU = 10.0
ALPHA1 = 0.5
ALPHA2 = 5.0
N1 = 500
N2 = 1500

CONTAMINATION_RATE = 0.01
CONTAMINATION_MULTIPLIER = 10.0

# Prespecify this grid before examining the final results.
# It extends beyond 0.01 so an old grid boundary cannot determine
# the selected value.
TAU_GRID = np.array(
    [
        0.0,
        0.000025,
        0.000050,
        0.000075,
        0.000100,
        0.000250,
        0.000500,
        0.000750,
        0.001000,
        0.001250,
        0.001500,
        0.002000,
        0.003000,
        0.005000,
        0.007500,
        0.010000,
        0.012500,
        0.015000,
        0.020000,
        0.030000,
        0.050000,
        0.075000,
        0.100000,
        0.150000,
        0.200000,
    ],
    dtype=float,
)

# Primary clean-model performance constraint.
CLEAN_EFFICIENCY_FLOOR = 0.99

# Optional sensitivity values reported from the same training table.
EFFICIENCY_FLOORS_FOR_SENSITIVITY = (0.95, 0.975, 0.99)

MIN_FIT_SUCCESS_RATE = 0.98

# Maximum relative movement of the mean clean estimate from the
# tau=0 clean estimate across the paired training simulations.
MAX_TARGET_DRIFT = 0.05

# This program uses the one-standard-error rule as the primary selector.
SELECTION_RULE = "one_se"

# False gives a manageable trial run. Use True for manuscript results.
PUBLICATION_MODE = False

if PUBLICATION_MODE:
    N_TRAIN_DATASETS = 1000
    N_TEST_DATASETS = 2000
    N_SELECTION_BOOTSTRAPS = 1000
else:
    N_TRAIN_DATASETS = 80
    N_TEST_DATASETS = 120
    N_SELECTION_BOOTSTRAPS = 200

OUTPUT_DIR = Path("common_mean_gamma_1pct_PR1SE_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)


# ============================================================
# 2. Estimators from the uploaded notebook
# ============================================================

def est_equal(x, y):
    """Unweighted average of the two sample means."""
    return float(0.5 * (np.mean(x) + np.mean(y)))


def est_pooled(x, y):
    """Sample-size-weighted pooled mean."""
    n1 = len(x)
    n2 = len(y)

    return float(
        (n1 * np.mean(x) + n2 * np.mean(y))
        / (n1 + n2)
    )


def est_plugin(x, y):
    """Plug-in inverse-variance weighted common mean."""
    n1 = len(x)
    n2 = len(y)

    mx = np.mean(x)
    my = np.mean(y)

    vx = np.var(x, ddof=1)
    vy = np.var(y, ddof=1)

    eps = 1.0e-12

    px = n1 / max(vx, eps)
    py = n2 / max(vy, eps)

    return float((px * mx + py * my) / (px + py))


def est_mom(x, y):
    """Method-of-moments shape-weighted common mean."""
    n1 = len(x)
    n2 = len(y)

    mx = np.mean(x)
    my = np.mean(y)

    vx = np.var(x, ddof=1)
    vy = np.var(y, ddof=1)

    eps = 1.0e-12

    alpha1_mom = mx * mx / max(vx, eps)
    alpha2_mom = my * my / max(vy, eps)

    w1 = n1 * alpha1_mom
    w2 = n2 * alpha2_mom

    return float((w1 * mx + w2 * my) / (w1 + w2))


# ============================================================
# 3. Constrained common-mean gamma MLE
# ============================================================

def gamma_logpdf_values(z, alpha, beta):
    """Gamma log density under shape alpha and scale beta."""
    z = np.asarray(z, dtype=float)

    return (
        (alpha - 1.0) * np.log(z)
        - z / beta
        - alpha * np.log(beta)
        - gammaln(alpha)
    )


def moment_start_values(x, y):
    """Log-scale method-of-moments starting values."""
    n1 = len(x)
    n2 = len(y)

    mx = np.mean(x)
    my = np.mean(y)
    vx = np.var(x, ddof=1)
    vy = np.var(y, ddof=1)

    eps = 1.0e-10

    mu0 = max(
        (n1 * mx + n2 * my) / (n1 + n2),
        eps,
    )
    alpha10 = max(mx * mx / max(vx, eps), 0.05)
    alpha20 = max(my * my / max(vy, eps), 0.05)

    return np.log([mu0, alpha10, alpha20])


def winsor_start_values(x, y, lower, upper):
    """Robust optimization start after within-group winsorization."""
    xw = np.clip(
        x,
        np.quantile(x, lower),
        np.quantile(x, upper),
    )
    yw = np.clip(
        y,
        np.quantile(y, lower),
        np.quantile(y, upper),
    )

    return moment_start_values(xw, yw)


def neg_loglik_common_mean(theta, x, y):
    """Negative log likelihood under the common-mean gamma model."""
    log_mu, log_alpha1, log_alpha2 = theta

    mu = np.exp(log_mu)
    alpha1 = np.exp(log_alpha1)
    alpha2 = np.exp(log_alpha2)

    beta1 = mu / alpha1
    beta2 = mu / alpha2

    value = -(
        np.sum(gamma_logpdf_values(x, alpha1, beta1))
        + np.sum(gamma_logpdf_values(y, alpha2, beta2))
    )

    return float(value) if np.isfinite(value) else 1.0e100


def mle_common_mean_params(x, y):
    """
    Constrained MLE of (mu, alpha1, alpha2).

    A finite MOM fallback is returned if numerical optimization fails.
    """
    theta0 = moment_start_values(x, y)

    bounds = [
        (np.log(1.0e-8), np.log(1.0e8)),
        (np.log(1.0e-6), np.log(1.0e6)),
        (np.log(1.0e-6), np.log(1.0e6)),
    ]

    res = minimize(
        neg_loglik_common_mean,
        theta0,
        args=(x, y),
        method="L-BFGS-B",
        bounds=bounds,
        options={
            "maxiter": 400,
            "ftol": 1.0e-11,
        },
    )

    if res.success and np.all(np.isfinite(res.x)):
        return (
            float(np.exp(res.x[0])),
            float(np.exp(res.x[1])),
            float(np.exp(res.x[2])),
            True,
        )

    mu0, alpha10, alpha20 = np.exp(theta0)

    return (
        float(mu0),
        float(alpha10),
        float(alpha20),
        False,
    )


def est_better_component_plugin(x, y, mle_result=None):
    """
    Select the sample mean from the component with larger plug-in
    information n_g * alpha_g.
    """
    if mle_result is None:
        mle_result = mle_common_mean_params(x, y)

    _, alpha1_hat, alpha2_hat, _ = mle_result

    information1 = len(x) * alpha1_hat
    information2 = len(y) * alpha2_hat

    if information1 >= information2:
        return float(np.mean(x))

    return float(np.mean(y))


# ============================================================
# 4. Stable ADPD objective
# ============================================================

def dpd_integral_gamma(alpha, beta, tau):
    """
    Integral of f^(1+tau) for a gamma density.

    The integral exists when
        (1 + tau) * alpha - tau > 0.
    """
    kappa = (1.0 + tau) * alpha - tau

    if alpha <= 0.0 or beta <= 0.0 or kappa <= 0.0:
        return np.inf

    log_value = (
        gammaln(kappa)
        - (1.0 + tau) * gammaln(alpha)
        - tau * np.log(beta)
        - kappa * np.log1p(tau)
    )

    if not np.isfinite(log_value) or log_value > 700.0:
        return np.inf

    return float(np.exp(log_value))


def dpd_group_objective(z, alpha, beta, tau):
    """
    Numerically stable centered DPD objective contribution.

    The empirical term uses
        sum(expm1(tau * log f_i))
      = sum(f_i^tau - 1).

    The omitted constant does not depend on the parameters.
    """
    integral = dpd_integral_gamma(alpha, beta, tau)

    if not np.isfinite(integral):
        return 1.0e100

    log_f = gamma_logpdf_values(z, alpha, beta)
    tau_log_f = np.clip(tau * log_f, -745.0, 700.0)

    centered_power_sum = np.sum(np.expm1(tau_log_f))

    value = len(z) * integral
    value -= ((1.0 + tau) / tau) * centered_power_sum

    return float(value) if np.isfinite(value) else 1.0e100


def dpd_common_mean_objective(theta, x, y, tau):
    """Two-group common-mean DPD objective."""
    log_mu, log_alpha1, log_alpha2 = theta

    mu = np.exp(log_mu)
    alpha1 = np.exp(log_alpha1)
    alpha2 = np.exp(log_alpha2)

    beta1 = mu / alpha1
    beta2 = mu / alpha2

    value = (
        dpd_group_objective(x, alpha1, beta1, tau)
        + dpd_group_objective(y, alpha2, beta2, tau)
    )

    return float(value) if np.isfinite(value) else 1.0e100


def est_adpd_common_mean(
    x,
    y,
    tau,
    mle_result=None,
    extra_start=None,
):
    """
    Fit ADPD at a fixed tau.

    Tau = 0 returns the constrained MLE, the likelihood-limit member
    of the DPD family.
    """
    if mle_result is None:
        mle_result = mle_common_mean_params(x, y)

    if tau <= 1.0e-12:
        return mle_result

    mu_mle, alpha1_mle, alpha2_mle, _ = mle_result

    starts = [
        np.log(
            [
                max(mu_mle, 1.0e-8),
                max(alpha1_mle, 1.0e-4),
                max(alpha2_mle, 1.0e-4),
            ]
        ),
        moment_start_values(x, y),
        winsor_start_values(x, y, 0.05, 0.95),
        winsor_start_values(x, y, 0.10, 0.90),
    ]

    if extra_start is not None:
        starts.insert(0, np.asarray(extra_start, dtype=float))

    bounds = [
        (np.log(1.0e-8), np.log(1.0e8)),
        (np.log(1.0e-4), np.log(1.0e6)),
        (np.log(1.0e-4), np.log(1.0e6)),
    ]

    best_value = np.inf
    best_theta = None

    for theta0 in starts:
        res = minimize(
            dpd_common_mean_objective,
            theta0,
            args=(x, y, tau),
            method="L-BFGS-B",
            bounds=bounds,
            options={
                "maxiter": 400,
                "ftol": 1.0e-11,
            },
        )

        if not (
            res.success
            and np.all(np.isfinite(res.x))
        ):
            continue

        value = dpd_common_mean_objective(
            res.x,
            x,
            y,
            tau,
        )

        if np.isfinite(value) and value < best_value:
            best_value = value
            best_theta = res.x.copy()

    if best_theta is None:
        return np.nan, np.nan, np.nan, False

    return (
        float(np.exp(best_theta[0])),
        float(np.exp(best_theta[1])),
        float(np.exp(best_theta[2])),
        True,
    )


def fit_adpd_grid(x, y, tau_grid):
    """Fit every candidate tau to one dataset using warm starts."""
    mle_result = mle_common_mean_params(x, y)

    rows = []
    previous_theta = None

    for tau in np.sort(np.asarray(tau_grid, dtype=float)):
        fit = est_adpd_common_mean(
            x=x,
            y=y,
            tau=float(tau),
            mle_result=mle_result,
            extra_start=previous_theta,
        )

        mu_hat, alpha1_hat, alpha2_hat, ok = fit

        rows.append(
            {
                "tau": float(tau),
                "estimate": mu_hat,
                "alpha1_hat": alpha1_hat,
                "alpha2_hat": alpha2_hat,
                "fit_success": bool(ok),
            }
        )

        if ok and tau > 0.0:
            previous_theta = np.log(
                [
                    max(mu_hat, 1.0e-8),
                    max(alpha1_hat, 1.0e-4),
                    max(alpha2_hat, 1.0e-4),
                ]
            )

    return pd.DataFrame(rows)


# ============================================================
# 5. Paired clean and contaminated data generation
# ============================================================

def generate_paired_data(rng):
    """
    Generate a clean common-mean gamma dataset and its contaminated copy.

    Contamination is applied independently to each observation by
    multiplying it by CONTAMINATION_MULTIPLIER with probability
    CONTAMINATION_RATE.
    """
    x_clean = rng.gamma(
        shape=ALPHA1,
        scale=TRUE_MU / ALPHA1,
        size=N1,
    )

    y_clean = rng.gamma(
        shape=ALPHA2,
        scale=TRUE_MU / ALPHA2,
        size=N2,
    )

    x_contaminated = x_clean.copy()
    y_contaminated = y_clean.copy()

    mask_x = rng.random(N1) < CONTAMINATION_RATE
    mask_y = rng.random(N2) < CONTAMINATION_RATE

    x_contaminated[mask_x] *= CONTAMINATION_MULTIPLIER
    y_contaminated[mask_y] *= CONTAMINATION_MULTIPLIER

    return {
        "x_clean": x_clean,
        "y_clean": y_clean,
        "x_contaminated": x_contaminated,
        "y_contaminated": y_contaminated,
        "n_contaminated_x": int(mask_x.sum()),
        "n_contaminated_y": int(mask_y.sum()),
    }


def make_independent_seeds(n, seed):
    """Create independent deterministic seeds for synthetic datasets."""
    rng = np.random.default_rng(seed)

    return rng.integers(
        low=1,
        high=np.iinfo(np.int32).max,
        size=n,
        dtype=np.int64,
    )


# ============================================================
# 6. Training: ADPD tau calibration
# ============================================================

def build_training_results(dataset_seeds, tau_grid):
    """
    Fit the ADPD tau grid to paired clean and contaminated training data.
    """
    rows = []

    for rep, dataset_seed in enumerate(dataset_seeds):
        if rep == 0 or (rep + 1) % max(1, len(dataset_seeds) // 10) == 0:
            print(
                f"training pair {rep + 1} "
                f"of {len(dataset_seeds)}"
            )

        rng = np.random.default_rng(int(dataset_seed))
        generated = generate_paired_data(rng)

        clean_fits = fit_adpd_grid(
            generated["x_clean"],
            generated["y_clean"],
            tau_grid,
        ).rename(
            columns={
                "estimate": "clean_estimate",
                "fit_success": "clean_fit_success",
                "alpha1_hat": "clean_alpha1_hat",
                "alpha2_hat": "clean_alpha2_hat",
            }
        )

        contaminated_fits = fit_adpd_grid(
            generated["x_contaminated"],
            generated["y_contaminated"],
            tau_grid,
        ).rename(
            columns={
                "estimate": "contaminated_estimate",
                "fit_success": "contaminated_fit_success",
                "alpha1_hat": "contaminated_alpha1_hat",
                "alpha2_hat": "contaminated_alpha2_hat",
            }
        )

        merged = clean_fits.merge(
            contaminated_fits,
            on="tau",
            how="inner",
        )

        for row in merged.itertuples(index=False):
            clean_estimate = float(row.clean_estimate)
            contaminated_estimate = float(row.contaminated_estimate)

            rows.append(
                {
                    "rep": rep,
                    "dataset_seed": int(dataset_seed),
                    "tau": float(row.tau),
                    "clean_estimate": clean_estimate,
                    "contaminated_estimate": contaminated_estimate,
                    "clean_error": clean_estimate - TRUE_MU,
                    "contaminated_error": contaminated_estimate - TRUE_MU,
                    "clean_squared_error": (
                        clean_estimate - TRUE_MU
                    ) ** 2,
                    "contaminated_squared_error": (
                        contaminated_estimate - TRUE_MU
                    ) ** 2,
                    "clean_fit_success": bool(row.clean_fit_success),
                    "contaminated_fit_success": bool(
                        row.contaminated_fit_success
                    ),
                    "n_contaminated_x": generated["n_contaminated_x"],
                    "n_contaminated_y": generated["n_contaminated_y"],
                }
            )

    return pd.DataFrame(rows)


def summarize_calibration_risk(training_results):
    """
    Estimate clean and contaminated risks, Monte Carlo uncertainty,
    clean efficiency, numerical success, and target drift for each tau.

    In simulation, target drift is measured by comparing the Monte Carlo
    mean clean estimate at tau with the Monte Carlo mean clean estimate at
    tau=0. This is the simulation analogue of the full-development-sample
    DPD--MLE drift safeguard used in the real-data analysis.
    """
    rows = []
    n_reps = training_results["rep"].nunique()

    for tau, group in training_results.groupby("tau"):
        valid = group[
            group["clean_fit_success"]
            & group["contaminated_fit_success"]
            & np.isfinite(group["clean_estimate"])
            & np.isfinite(group["contaminated_estimate"])
        ].copy()

        n = len(valid)

        if n == 0:
            rows.append(
                {
                    "tau": float(tau),
                    "training_n": 0,
                    "fit_success_rate": 0.0,
                }
            )
            continue

        clean_estimates = valid[
            "clean_estimate"
        ].to_numpy(dtype=float)

        contaminated_estimates = valid[
            "contaminated_estimate"
        ].to_numpy(dtype=float)

        clean_squared_errors = valid[
            "clean_squared_error"
        ].to_numpy(dtype=float)

        contaminated_squared_errors = valid[
            "contaminated_squared_error"
        ].to_numpy(dtype=float)

        clean_mse = float(
            np.mean(clean_squared_errors)
        )

        contaminated_mse = float(
            np.mean(contaminated_squared_errors)
        )

        if n >= 2:
            clean_variance = float(
                np.var(
                    clean_estimates,
                    ddof=1,
                )
            )

            contaminated_variance = float(
                np.var(
                    contaminated_estimates,
                    ddof=1,
                )
            )

            clean_mse_mcse = float(
                np.std(
                    clean_squared_errors,
                    ddof=1,
                )
                / np.sqrt(n)
            )

            contaminated_mse_mcse = float(
                np.std(
                    contaminated_squared_errors,
                    ddof=1,
                )
                / np.sqrt(n)
            )
        else:
            clean_variance = np.nan
            contaminated_variance = np.nan
            clean_mse_mcse = np.nan
            contaminated_mse_mcse = np.nan

        rows.append(
            {
                "tau": float(tau),
                "training_n": n,
                "fit_success_rate": n / n_reps,
                "clean_mean_estimate": float(
                    np.mean(clean_estimates)
                ),
                "clean_bias": float(
                    np.mean(
                        clean_estimates
                        - TRUE_MU
                    )
                ),
                "clean_variance": (
                    clean_variance
                ),
                "clean_estimate_sd": float(
                    np.sqrt(
                        clean_variance
                    )
                )
                if np.isfinite(
                    clean_variance
                )
                else np.nan,
                "clean_mse": clean_mse,
                "clean_mse_mcse": (
                    clean_mse_mcse
                ),
                "clean_rmse": float(
                    np.sqrt(clean_mse)
                ),
                "contaminated_mean_estimate": float(
                    np.mean(
                        contaminated_estimates
                    )
                ),
                "contaminated_bias": float(
                    np.mean(
                        contaminated_estimates
                        - TRUE_MU
                    )
                ),
                "contaminated_variance": (
                    contaminated_variance
                ),
                "contaminated_mse": (
                    contaminated_mse
                ),
                "contaminated_mse_mcse": (
                    contaminated_mse_mcse
                ),
                "contaminated_rmse": float(
                    np.sqrt(
                        contaminated_mse
                    )
                ),
            }
        )

    summary = (
        pd.DataFrame(rows)
        .sort_values("tau")
        .reset_index(drop=True)
    )

    tau0 = summary.loc[
        np.isclose(
            summary["tau"],
            0.0,
        )
    ]

    if len(tau0) != 1:
        raise RuntimeError(
            "The tau=0 clean reference row is "
            "missing or duplicated."
        )

    clean_mse_tau0 = float(
        tau0.iloc[0]["clean_mse"]
    )

    clean_variance_tau0 = float(
        tau0.iloc[0][
            "clean_variance"
        ]
    )

    clean_mean_tau0 = float(
        tau0.iloc[0][
            "clean_mean_estimate"
        ]
    )

    summary[
        "clean_mse_efficiency"
    ] = (
        clean_mse_tau0
        / summary["clean_mse"]
    )

    summary[
        "clean_variance_efficiency"
    ] = (
        clean_variance_tau0
        / summary["clean_variance"]
    )

    summary[
        "target_drift_from_tau0"
    ] = (
        np.abs(
            summary[
                "clean_mean_estimate"
            ]
            - clean_mean_tau0
        )
        / abs(clean_mean_tau0)
    )

    summary[
        "feasible_primary"
    ] = (
        summary[
            "clean_mse_efficiency"
        ]
        >= CLEAN_EFFICIENCY_FLOOR
    ) & (
        summary[
            "fit_success_rate"
        ]
        >= MIN_FIT_SUCCESS_RATE
    ) & (
        summary[
            "target_drift_from_tau0"
        ]
        <= MAX_TARGET_DRIFT
    )

    return summary


def select_tau_one_se(
    calibration_summary,
    efficiency_floor=CLEAN_EFFICIENCY_FLOOR,
):
    """
    Clean-precision-refined one-standard-error selection.

    1. Enforce clean relative-MSE efficiency, numerical-success, and
       target-drift constraints.
    2. Find the feasible contaminated-MSE minimizer.
    3. Form the robust one-standard-error set.
    4. Within that set, choose the smallest clean estimate SD.
       Target drift and tau are lexicographic tie-breakers.
    """
    feasible = calibration_summary[
        (
            calibration_summary[
                "clean_mse_efficiency"
            ]
            >= efficiency_floor
        )
        & (
            calibration_summary[
                "fit_success_rate"
            ]
            >= MIN_FIT_SUCCESS_RATE
        )
        & (
            calibration_summary[
                "target_drift_from_tau0"
            ]
            <= MAX_TARGET_DRIFT
        )
        & np.isfinite(
            calibration_summary[
                "contaminated_mse"
            ]
        )
        & np.isfinite(
            calibration_summary[
                "contaminated_mse_mcse"
            ]
        )
        & np.isfinite(
            calibration_summary[
                "clean_estimate_sd"
            ]
        )
    ].copy()

    if feasible.empty:
        raise RuntimeError(
            "No tau satisfies the clean-efficiency, "
            "fit-success, and target-drift constraints."
        )

    minimum_row = feasible.loc[
        feasible[
            "contaminated_mse"
        ].idxmin()
    ]

    exact_minimizer = float(
        minimum_row["tau"]
    )

    minimum_mse = float(
        minimum_row[
            "contaminated_mse"
        ]
    )

    minimum_mse_mcse = float(
        minimum_row[
            "contaminated_mse_mcse"
        ]
    )

    one_se_threshold = (
        minimum_mse
        + minimum_mse_mcse
    )

    one_se_candidates = (
        feasible.loc[
            feasible[
                "contaminated_mse"
            ]
            <= one_se_threshold
        ]
        .sort_values(
            [
                "clean_estimate_sd",
                "target_drift_from_tau0",
                "tau",
            ],
            ascending=[
                True,
                True,
                True,
            ],
        )
    )

    selected_row = (
        one_se_candidates.iloc[0]
    )

    selected_tau = float(
        selected_row["tau"]
    )

    return {
        "selection_rule": (
            "precision_refined_one_se"
        ),
        "clean_efficiency_floor": (
            efficiency_floor
        ),
        "maximum_target_drift": (
            MAX_TARGET_DRIFT
        ),
        "selected_tau": selected_tau,
        "exact_feasible_minimizer": (
            exact_minimizer
        ),
        "minimum_feasible_contaminated_mse": (
            minimum_mse
        ),
        "minimum_contaminated_mse_mcse": (
            minimum_mse_mcse
        ),
        "one_se_threshold": (
            one_se_threshold
        ),
        "selected_clean_mse_efficiency": float(
            selected_row[
                "clean_mse_efficiency"
            ]
        ),
        "selected_clean_variance_efficiency": float(
            selected_row[
                "clean_variance_efficiency"
            ]
        ),
        "selected_clean_estimate_sd": float(
            selected_row[
                "clean_estimate_sd"
            ]
        ),
        "selected_target_drift": float(
            selected_row[
                "target_drift_from_tau0"
            ]
        ),
        "selected_contaminated_mse": float(
            selected_row[
                "contaminated_mse"
            ]
        ),
        "selected_contaminated_mse_mcse": float(
            selected_row[
                "contaminated_mse_mcse"
            ]
        ),
        "selected_fit_success_rate": float(
            selected_row[
                "fit_success_rate"
            ]
        ),
        "number_feasible": int(
            len(feasible)
        ),
        "number_within_one_se": int(
            len(one_se_candidates)
        ),
    }


def build_efficiency_floor_sensitivity(calibration_summary):
    """Repeat the precision-refined one-SE selection for prespecified efficiency floors."""
    rows = []

    for floor in EFFICIENCY_FLOORS_FOR_SENSITIVITY:
        try:
            selection = select_tau_one_se(
                calibration_summary,
                efficiency_floor=float(floor),
            )
            rows.append(selection)
        except RuntimeError as exc:
            rows.append(
                {
                    "selection_rule": "precision_refined_one_se",
                    "clean_efficiency_floor": float(floor),
                    "selected_tau": np.nan,
                    "error": str(exc),
                }
            )

    return pd.DataFrame(rows)


def bootstrap_selection_stability(
    training_results,
    n_bootstrap,
    seed,
):
    """
    Bootstrap whole paired training replications and repeat the complete
    primary one-SE selection.
    """
    rng = np.random.default_rng(seed)
    rep_ids = np.sort(training_results["rep"].unique())

    selected_values = []
    failures = 0

    for b in range(n_bootstrap):
        sampled_rep_ids = rng.choice(
            rep_ids,
            size=len(rep_ids),
            replace=True,
        )

        pieces = []

        for new_rep, old_rep in enumerate(sampled_rep_ids):
            part = training_results[
                training_results["rep"] == old_rep
            ].copy()

            part["rep"] = new_rep
            pieces.append(part)

        bootstrap_data = pd.concat(
            pieces,
            ignore_index=True,
        )

        try:
            bootstrap_summary = summarize_calibration_risk(
                bootstrap_data
            )
            bootstrap_selection = select_tau_one_se(
                bootstrap_summary,
                efficiency_floor=CLEAN_EFFICIENCY_FLOOR,
            )
            selected_values.append(
                bootstrap_selection["selected_tau"]
            )
        except Exception:
            failures += 1

    counts = (
        pd.Series(selected_values, name="selected_tau")
        .value_counts()
        .sort_index()
        .rename_axis("selected_tau")
        .reset_index(name="count")
    )

    if len(selected_values) > 0:
        counts["selection_rate"] = (
            counts["count"] / len(selected_values)
        )
    else:
        counts["selection_rate"] = np.nan

    return counts, failures


# ============================================================
# 7. Held-out evaluation of all estimators
# ============================================================

def compute_all_estimators(x, y, selected_tau):
    """
    Compute every estimator requested in the uploaded notebook.

    The ADPD tuning value is frozen from the training calibration.
    """
    estimates = {}
    success = {}

    estimates["Equal"] = est_equal(x, y)
    success["Equal"] = True

    estimates["Pooled"] = est_pooled(x, y)
    success["Pooled"] = True

    mle_result = mle_common_mean_params(x, y)
    mu_mle, _, _, mle_ok = mle_result

    estimates["BetterComponentPlugIn"] = (
        est_better_component_plugin(
            x,
            y,
            mle_result=mle_result,
        )
    )
    success["BetterComponentPlugIn"] = bool(mle_ok)

    estimates["Plugin"] = est_plugin(x, y)
    success["Plugin"] = True

    estimates["MOM"] = est_mom(x, y)
    success["MOM"] = True

    estimates["MLE"] = float(mu_mle)
    success["MLE"] = bool(mle_ok)

    adpd_fit = est_adpd_common_mean(
        x=x,
        y=y,
        tau=selected_tau,
        mle_result=mle_result,
    )

    estimates["ADPD-OneSE"] = float(adpd_fit[0])
    success["ADPD-OneSE"] = bool(adpd_fit[3])

    return estimates, success


def evaluate_all_estimators_on_test(
    dataset_seeds,
    selected_tau,
):
    """
    Evaluate all estimators on independent paired clean/contaminated data.
    """
    rows = []

    for rep, dataset_seed in enumerate(dataset_seeds):
        if rep == 0 or (rep + 1) % max(1, len(dataset_seeds) // 10) == 0:
            print(
                f"test pair {rep + 1} "
                f"of {len(dataset_seeds)}"
            )

        rng = np.random.default_rng(int(dataset_seed))
        generated = generate_paired_data(rng)

        scenarios = [
            (
                "clean",
                generated["x_clean"],
                generated["y_clean"],
            ),
            (
                "contaminated",
                generated["x_contaminated"],
                generated["y_contaminated"],
            ),
        ]

        for scenario, x, y in scenarios:
            estimates, success = compute_all_estimators(
                x=x,
                y=y,
                selected_tau=selected_tau,
            )

            for estimator, estimate in estimates.items():
                estimate = float(estimate)

                rows.append(
                    {
                        "rep": rep,
                        "dataset_seed": int(dataset_seed),
                        "scenario": scenario,
                        "estimator": estimator,
                        "selected_tau": (
                            selected_tau
                            if estimator == "ADPD-OneSE"
                            else np.nan
                        ),
                        "estimate": estimate,
                        "error": estimate - TRUE_MU,
                        "squared_error": (
                            estimate - TRUE_MU
                        ) ** 2,
                        "absolute_error": abs(
                            estimate - TRUE_MU
                        ),
                        "fit_success": bool(success[estimator]),
                        "n_contaminated_x": (
                            generated["n_contaminated_x"]
                        ),
                        "n_contaminated_y": (
                            generated["n_contaminated_y"]
                        ),
                    }
                )

    return pd.DataFrame(rows)


def summarize_test_estimators(test_results):
    """Held-out performance of all estimators by scenario."""
    rows = []
    total_reps = test_results["rep"].nunique()

    for (scenario, estimator), group in test_results.groupby(
        ["scenario", "estimator"]
    ):
        valid = group[
            group["fit_success"]
            & np.isfinite(group["estimate"])
        ].copy()

        n = len(valid)

        if n == 0:
            continue

        estimates = valid["estimate"].to_numpy(dtype=float)
        errors = valid["error"].to_numpy(dtype=float)
        squared_errors = valid[
            "squared_error"
        ].to_numpy(dtype=float)

        mse = float(np.mean(squared_errors))

        if n >= 2:
            variance = float(
                np.var(estimates, ddof=1)
            )
            bias_mcse = float(
                np.std(errors, ddof=1) / np.sqrt(n)
            )
            mse_mcse = float(
                np.std(squared_errors, ddof=1)
                / np.sqrt(n)
            )
        else:
            variance = np.nan
            bias_mcse = np.nan
            mse_mcse = np.nan

        rows.append(
            {
                "scenario": scenario,
                "estimator": estimator,
                "selected_tau": (
                    float(valid["selected_tau"].dropna().iloc[0])
                    if estimator == "ADPD-OneSE"
                    else np.nan
                ),
                "test_n": n,
                "mean_estimate": np.mean(estimates),
                "bias": np.mean(errors),
                "bias_mcse": bias_mcse,
                "variance": variance,
                "mse": mse,
                "mse_mcse": mse_mcse,
                "rmse": np.sqrt(mse),
                "mae": valid["absolute_error"].mean(),
                "fit_success_rate": n / total_reps,
            }
        )

    summary = pd.DataFrame(rows)

    summary["mse_rank_within_scenario"] = (
        summary.groupby("scenario")["mse"]
        .rank(method="min")
        .astype(int)
    )

    return summary.sort_values(
        ["scenario", "mse_rank_within_scenario", "estimator"]
    ).reset_index(drop=True)



# ============================================================
# 8. Figures
# ============================================================

def save_calibration_figures(
    calibration_summary,
    selection,
):
    """Save clean-efficiency and contaminated-risk figures."""
    selected_tau = selection["selected_tau"]
    threshold = selection["one_se_threshold"]

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.plot(
        calibration_summary["tau"],
        calibration_summary["clean_mse_efficiency"],
        marker="o",
    )
    ax.axhline(
        CLEAN_EFFICIENCY_FLOOR,
        linestyle="--",
        label=(
            "clean MSE efficiency floor = "
            f"{CLEAN_EFFICIENCY_FLOOR:.3f}"
        ),
    )
    ax.axvline(
        selected_tau,
        linestyle=":",
        label=f"one-SE tau = {selected_tau:.6f}",
    )
    ax.set_xlabel("Tau")
    ax.set_ylabel("Clean relative MSE efficiency")
    ax.set_title("Clean-model efficiency constraint")
    ax.legend()
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / "clean_mse_efficiency_by_tau.png",
        dpi=300,
    )
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 5))

    feasible = calibration_summary[
        "feasible_primary"
    ]

    ax.plot(
        calibration_summary["tau"],
        calibration_summary["contaminated_mse"],
        marker="o",
        label="all tau values",
    )
    ax.scatter(
        calibration_summary.loc[feasible, "tau"],
        calibration_summary.loc[
            feasible,
            "contaminated_mse",
        ],
        label="feasible tau values",
    )
    ax.axhline(
        threshold,
        linestyle=":",
        label="one-SE MSE threshold",
    )
    ax.axvline(
        selected_tau,
        linestyle="--",
        label=f"selected tau = {selected_tau:.6f}",
    )
    ax.set_xlabel("Tau")
    ax.set_ylabel("Contaminated training MSE")
    ax.set_title("One-SE ADPD calibration")
    ax.legend()
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / "one_se_contaminated_mse_by_tau.png",
        dpi=300,
    )
    plt.close(fig)


def save_test_mse_figures(test_summary):
    """Save separate clean and contaminated test-MSE figures."""
    for scenario in ["clean", "contaminated"]:
        table = test_summary[
            test_summary["scenario"] == scenario
        ].sort_values("mse")

        fig, ax = plt.subplots(figsize=(9, 5))

        ax.bar(
            table["estimator"],
            table["mse"],
        )
        ax.set_xlabel("Estimator")
        ax.set_ylabel("Held-out test MSE")
        ax.set_title(
            f"Held-out {scenario} test MSE"
        )
        ax.tick_params(
            axis="x",
            rotation=35,
        )
        fig.tight_layout()
        fig.savefig(
            OUTPUT_DIR / f"{scenario}_test_mse_by_estimator.png",
            dpi=300,
        )
        plt.close(fig)


# ============================================================
# 9. Main program
# ============================================================

def main():
    print("ADPD ONE-SE CALIBRATION WITH ALL ESTIMATORS")
    print("=" * 82)
    print("True common mean =", TRUE_MU)
    print("Gamma shapes =", ALPHA1, "and", ALPHA2)
    print("Sample sizes =", N1, "and", N2)
    print("Contamination rate =", CONTAMINATION_RATE)
    print(
        "Contamination multiplier =",
        CONTAMINATION_MULTIPLIER,
    )
    print(
        "Primary clean MSE efficiency floor =",
        CLEAN_EFFICIENCY_FLOOR,
    )
    print("Selection rule =", SELECTION_RULE)
    print("Training dataset pairs =", N_TRAIN_DATASETS)
    print("Test dataset pairs =", N_TEST_DATASETS)
    print("Bootstrap recalibrations =", N_SELECTION_BOOTSTRAPS)
    print("Tau grid:")
    print(TAU_GRID)
    print()

    if SELECTION_RULE != "one_se":
        raise ValueError(
            "This program is designed to use SELECTION_RULE='one_se'."
        )

    train_seeds = make_independent_seeds(
        N_TRAIN_DATASETS,
        SEED,
    )
    test_seeds = make_independent_seeds(
        N_TEST_DATASETS,
        SEED + 1000000,
    )

    # ---------------- Training calibration ----------------
    training_results = build_training_results(
        dataset_seeds=train_seeds,
        tau_grid=TAU_GRID,
    )

    calibration_summary = summarize_calibration_risk(
        training_results
    )

    selection = select_tau_one_se(
        calibration_summary,
        efficiency_floor=CLEAN_EFFICIENCY_FLOOR,
    )

    selected_tau = float(selection["selected_tau"])

    sensitivity_table = build_efficiency_floor_sensitivity(
        calibration_summary
    )

    print()
    print("TRAINING ADPD CALIBRATION TABLE")
    print(calibration_summary.to_string(index=False))

    print()
    print("=" * 82)
    print(
        "PRECISION-REFINED ONE-SE SELECTED TAU =",
        f"{selected_tau:.6f}",
    )
    print(
        "Exact feasible MSE minimizer =",
        f"{selection['exact_feasible_minimizer']:.6f}",
    )
    print(
        "Minimum feasible contaminated MSE =",
        selection["minimum_feasible_contaminated_mse"],
    )
    print(
        "MCSE at the feasible minimum =",
        selection["minimum_contaminated_mse_mcse"],
    )
    print(
        "One-SE threshold =",
        selection["one_se_threshold"],
    )
    print(
        "Selected clean MSE efficiency =",
        selection["selected_clean_mse_efficiency"],
    )
    print(
        "Selected clean variance efficiency =",
        selection["selected_clean_variance_efficiency"],
    )
    print(
        "Selected clean estimate SD =",
        selection["selected_clean_estimate_sd"],
    )
    print(
        "Selected target drift =",
        selection["selected_target_drift"],
    )
    print("=" * 82)

    print()
    print("EFFICIENCY-FLOOR SENSITIVITY")
    print(sensitivity_table.to_string(index=False))

    # ---------------- Bootstrap selection stability ----------------
    stability_table, bootstrap_failures = (
        bootstrap_selection_stability(
            training_results=training_results,
            n_bootstrap=N_SELECTION_BOOTSTRAPS,
            seed=SEED + 500000,
        )
    )

    print()
    print("BOOTSTRAP PR-1SE SELECTION STABILITY")
    print(stability_table.to_string(index=False))
    print(
        "Bootstrap recalibration failures =",
        bootstrap_failures,
    )

    # ---------------- Independent test comparison ----------------
    test_details = evaluate_all_estimators_on_test(
        dataset_seeds=test_seeds,
        selected_tau=selected_tau,
    )

    test_summary = summarize_test_estimators(
        test_details
    )

    print()
    print("HELD-OUT TEST PERFORMANCE: ALL ESTIMATORS")
    print(test_summary.to_string(index=False))

    print()
    print(
        "FINAL PRECISION-REFINED ONE-SE SELECTED TAU =",
        f"{selected_tau:.6f}",
    )

    # ---------------- Save output ----------------
    training_results.to_csv(
        OUTPUT_DIR / "training_adpd_grid_details.csv",
        index=False,
    )

    calibration_summary.to_csv(
        OUTPUT_DIR / "training_adpd_calibration_summary.csv",
        index=False,
    )

    pd.DataFrame([selection]).to_csv(
        OUTPUT_DIR / "precision_refined_one_se_selected_tau.csv",
        index=False,
    )

    sensitivity_table.to_csv(
        OUTPUT_DIR / "efficiency_floor_sensitivity.csv",
        index=False,
    )

    stability_table.to_csv(
        OUTPUT_DIR / "precision_refined_one_se_selection_stability.csv",
        index=False,
    )

    test_details.to_csv(
        OUTPUT_DIR / "held_out_all_estimators_details.csv",
        index=False,
    )

    test_summary.to_csv(
        OUTPUT_DIR / "held_out_all_estimators_summary.csv",
        index=False,
    )

    settings = {
        "true_mu": TRUE_MU,
        "alpha1": ALPHA1,
        "alpha2": ALPHA2,
        "n1": N1,
        "n2": N2,
        "contamination_rate": CONTAMINATION_RATE,
        "contamination_multiplier": CONTAMINATION_MULTIPLIER,
        "tau_grid": TAU_GRID.tolist(),
        "clean_efficiency_floor": CLEAN_EFFICIENCY_FLOOR,
        "efficiency_floors_for_sensitivity": list(
            EFFICIENCY_FLOORS_FOR_SENSITIVITY
        ),
        "minimum_fit_success_rate": MIN_FIT_SUCCESS_RATE,
        "maximum_target_drift": MAX_TARGET_DRIFT,
        "selection_rule": SELECTION_RULE,
        "n_train_datasets": N_TRAIN_DATASETS,
        "n_test_datasets": N_TEST_DATASETS,
        "n_selection_bootstraps": N_SELECTION_BOOTSTRAPS,
        "selected_tau": selected_tau,
        "exact_feasible_minimizer": selection[
            "exact_feasible_minimizer"
        ],
        "one_se_threshold": selection["one_se_threshold"],
    }

    with open(
        OUTPUT_DIR / "simulation_settings.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(settings, file, indent=2)

    save_calibration_figures(
        calibration_summary=calibration_summary,
        selection=selection,
    )
    save_test_mse_figures(test_summary)

    print()
    print("Output folder:", OUTPUT_DIR.resolve())
    print("DONE")


if __name__ == "__main__":
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        main()


ADPD ONE-SE CALIBRATION WITH ALL ESTIMATORS
True common mean = 10.0
Gamma shapes = 0.5 and 5.0
Sample sizes = 500 and 1500
Contamination rate = 0.01
Contamination multiplier = 10.0
Primary clean MSE efficiency floor = 0.99
Selection rule = one_se
Training dataset pairs = 80
Test dataset pairs = 120
Bootstrap recalibrations = 200
Tau grid:
[0.00e+00 2.50e-05 5.00e-05 7.50e-05 1.00e-04 2.50e-04 5.00e-04 7.50e-04
 1.00e-03 1.25e-03 1.50e-03 2.00e-03 3.00e-03 5.00e-03 7.50e-03 1.00e-02
 1.25e-02 1.50e-02 2.00e-02 3.00e-02 5.00e-02 7.50e-02 1.00e-01 1.50e-01
 2.00e-01]

training pair 1 of 80
training pair 8 of 80
training pair 16 of 80
training pair 24 of 80
training pair 32 of 80
training pair 40 of 80
training pair 48 of 80
training pair 56 of 80
training pair 64 of 80
training pair 72 of 80
training pair 80 of 80

TRAINING ADPD CALIBRATION TABLE
     tau  training_n  fit_success_rate  clean_mean_estimate  clean_bias  clean_variance  clean_estimate_sd  clean_mse  clean_mse_mcse  clean_rms